In [ ]:
#Schritt 1 Bibliotheken und Einstellungen

In [5]:
import arcpy
import os

arcpy.env.overwriteOutput = True


# ============================================================
# BENUTZEREINSTELLUNGEN
# ============================================================

# Name des obersten KML-Layers im Bereich "Contents"
KML_LAYER_NAME = "GISMobil_ExportLWL_Export"


# Ausgabeordner
OUTPUT_FOLDER = (
      r"Q:\SNAP\SNAP_GIS_Analyse\Adam_Analyse"
    r"\\FTDB_Bearbeitung"
    r"\BSNetze"
)


# Arbeits-Geodatabase
OUTPUT_GDB_NAME = "BSNetze.gdb"


# Name der KML-Konvertierung
KML_CONVERSION_NAME = "kmltolayer"


# Ziel-Feature-Class in der Fremdtrassen-Datenbank
# Später den richtigen Pfad eintragen
TARGET_FEATURE_CLASS = r""


print("Einstellungen wurden geladen.")

Einstellungen wurden geladen.


In [ ]:
#Schritt 2 Ausgabeordner und Geodatabase vorbereiten

In [6]:
# ============================================================
# AUSGABEORDNER UND TEST.GDB VORBEREITEN
# ============================================================

if not os.path.isdir(OUTPUT_FOLDER):
    os.makedirs(
        OUTPUT_FOLDER,
        exist_ok=True
    )

OUTPUT_GDB = os.path.join(
    OUTPUT_FOLDER,
    OUTPUT_GDB_NAME
)

if not arcpy.Exists(OUTPUT_GDB):

    arcpy.management.CreateFileGDB(
        out_folder_path=OUTPUT_FOLDER,
        out_name=OUTPUT_GDB_NAME
    )

    print("Geodatabase wurde angelegt:", OUTPUT_GDB)

else:

    print("Geodatabase ist bereits vorhanden:", OUTPUT_GDB)


arcpy.env.workspace = OUTPUT_GDB

print("ArcPy-Workspace:", arcpy.env.workspace)

Geodatabase wurde angelegt: Q:\SNAP\SNAP_GIS_Analyse\Adam_Analyse\\FTDB_Bearbeitung\BSNetze\BSNetze.gdb
ArcPy-Workspace: Q:\SNAP\SNAP_GIS_Analyse\Adam_Analyse\\FTDB_Bearbeitung\BSNetze\BSNetze.gdb


In [ ]:
#Schritt 3 Geöffneten KML-Layer suchen

In [7]:
# ============================================================
# KML-LAYER IN DER AKTIVEN KARTE SUCHEN
# ============================================================

project = arcpy.mp.ArcGISProject("CURRENT")
active_map = project.activeMap

if active_map is None:
    raise RuntimeError(
        "In ArcGIS Pro ist keine Karte aktiv."
    )


matching_layers = [
    layer
    for layer in active_map.listLayers()
    if layer.name == KML_LAYER_NAME
]


if not matching_layers:

    available_layers = [
        layer.name
        for layer in active_map.listLayers()
    ]

    raise RuntimeError(
        f"Der Layer '{KML_LAYER_NAME}' wurde nicht gefunden.\n"
        f"Vorhandene Layer: {available_layers}"
    )


input_kml = matching_layers[0]

print("KML-Layer wurde gefunden:", input_kml.name)

KML-Layer wurde gefunden: GISMobil_ExportLWL_Export


In [ ]:
#Schritt 4 KML-Layer konvertieren

In [8]:
# ============================================================
# KML-LAYER KONVERTIEREN
# ============================================================

KML_GDB = os.path.join(
    OUTPUT_FOLDER,
    KML_CONVERSION_NAME + ".gdb"
)

KML_LAYER_FILE = os.path.join(
    OUTPUT_FOLDER,
    KML_CONVERSION_NAME + ".lyrx"
)


# Alte KML-Konvertierung löschen
for old_output in [
    KML_GDB,
    KML_LAYER_FILE
]:

    if arcpy.Exists(old_output):
        arcpy.management.Delete(old_output)
        print("Alte Ausgabe gelöscht:", old_output)


print("KML-Konvertierung wird gestartet ...")


arcpy.conversion.KMLToLayer(
    in_kml_file=input_kml,
    output_folder=OUTPUT_FOLDER,
    output_data=KML_CONVERSION_NAME
)


print("KML-Konvertierung abgeschlossen.")
print("KML-Geodatabase:", KML_GDB)
print("KML-Geodatabase vorhanden:", arcpy.Exists(KML_GDB))

KML-Konvertierung wird gestartet ...
KML-Konvertierung abgeschlossen.
KML-Geodatabase: Q:\SNAP\SNAP_GIS_Analyse\Adam_Analyse\\FTDB_Bearbeitung\BSNetze\kmltolayer.gdb
KML-Geodatabase vorhanden: True


In [ ]:
#Schritt 5 Polylinien exportieren

In [9]:
# ============================================================
# POLYLINIEN AUS DER KML-GEODATABASE EXPORTIEREN
# ============================================================

POLYLINE_SOURCE = os.path.join(
    KML_GDB,
    "Placemarks",
    "Polylines"
)

POLYLINE_OUTPUT = os.path.join(
    OUTPUT_GDB,
    "polylinetofeature"
)


print("Quelle der Polylinien:", POLYLINE_SOURCE)
print(
    "Polylinien vorhanden:",
    arcpy.Exists(POLYLINE_SOURCE)
)


if not arcpy.Exists(POLYLINE_SOURCE):
    raise RuntimeError(
        "Die Feature-Class 'Placemarks\\Polylines' "
        "wurde nicht gefunden."
    )


# Altes Ergebnis löschen
if arcpy.Exists(POLYLINE_OUTPUT):
    arcpy.management.Delete(POLYLINE_OUTPUT)


# Polylinien exportieren
arcpy.conversion.FeatureClassToFeatureClass(
    in_features=POLYLINE_SOURCE,
    out_path=OUTPUT_GDB,
    out_name="polylinetofeature",
    where_clause="",
    field_mapping="",
    config_keyword=""
)


polyline_count = int(
    arcpy.management.GetCount(
        POLYLINE_OUTPUT
    )[0]
)


print("Polylinien wurden exportiert.")
print("Ausgabe:", POLYLINE_OUTPUT)
print("Anzahl der Polylinien:", polyline_count)

Quelle der Polylinien: Q:\SNAP\SNAP_GIS_Analyse\Adam_Analyse\\FTDB_Bearbeitung\BSNetze\kmltolayer.gdb\Placemarks\Polylines
Polylinien vorhanden: True
Polylinien wurden exportiert.
Ausgabe: Q:\SNAP\SNAP_GIS_Analyse\Adam_Analyse\\FTDB_Bearbeitung\BSNetze\BSNetze.gdb\polylinetofeature
Anzahl der Polylinien: 53769


In [ ]:
#Schritt 6 Nach EPSG:25832 projizieren

In [10]:
# ============================================================
# PROJEKTION NACH ETRS89 / UTM ZONE 32N
# EPSG:25832
# ============================================================

PROJECTED_OUTPUT = os.path.join(
    OUTPUT_GDB,
    "polylinetofeature_Projected"
)


# Altes Ergebnis löschen
if arcpy.Exists(PROJECTED_OUTPUT):
    arcpy.management.Delete(PROJECTED_OUTPUT)


source_coordinate_system = arcpy.Describe(
    POLYLINE_OUTPUT
).spatialReference

target_coordinate_system = arcpy.SpatialReference(
    25832
)


print(
    "Quell-Koordinatensystem:",
    source_coordinate_system.name
)

print(
    "Ziel-Koordinatensystem:",
    target_coordinate_system.name
)


transformations = arcpy.ListTransformations(
    source_coordinate_system,
    target_coordinate_system
)


if transformations:
    transformation = transformations[0]
else:
    transformation = ""


print(
    "Verwendete Transformation:",
    transformation if transformation else "Keine"
)


arcpy.management.Project(
    in_dataset=POLYLINE_OUTPUT,
    out_dataset=PROJECTED_OUTPUT,
    out_coor_system=target_coordinate_system,
    transform_method=transformation
)


projected_count = int(
    arcpy.management.GetCount(
        PROJECTED_OUTPUT
    )[0]
)


print("Projektion abgeschlossen.")
print("Ausgabe:", PROJECTED_OUTPUT)
print("Anzahl der Features:", projected_count)

Quell-Koordinatensystem: GCS_WGS_1984
Ziel-Koordinatensystem: ETRS_1989_UTM_Zone_32N
Verwendete Transformation: ETRS_1989_To_WGS_1984
Projektion abgeschlossen.
Ausgabe: Q:\SNAP\SNAP_GIS_Analyse\Adam_Analyse\\FTDB_Bearbeitung\BSNetze\BSNetze.gdb\polylinetofeature_Projected
Anzahl der Features: 53769


In [ ]:
#Schritt 7 Nicht benötigte Felder löschen und neue Felder anlegen

In [11]:
# ============================================================
# FELDER VORBEREITEN
# ============================================================

fc = PROJECTED_OUTPUT


# ------------------------------------------------------------
# Vorhandene Felder anzeigen
# ------------------------------------------------------------

print("Vorhandene Felder:\n")

for field in arcpy.ListFields(fc):

    print(
        f"{field.name:30} "
        f"{field.type:12} "
        f"required={field.required}"
    )


# ------------------------------------------------------------
# Nicht benötigte Felder löschen
# ------------------------------------------------------------

KEEP_FIELDS = {
    "folderpath",
    "name"
}

fields_to_delete = []

for field in arcpy.ListFields(fc):

    if field.required:
        continue

    if field.name.lower() in KEEP_FIELDS:
        continue

    fields_to_delete.append(field.name)


print("\nZu löschende Felder:", fields_to_delete)


if fields_to_delete:

    arcpy.management.DeleteField(
        in_table=fc,
        drop_field=fields_to_delete
    )


# ------------------------------------------------------------
# Neue Datenbankfelder anlegen
# ------------------------------------------------------------

fields_to_add = [
    ("LR_NR", "TEXT", 50),
    ("LR_TYP", "TEXT", 100),
    ("LR_PKC", "TEXT", 50),
    ("LR_PEC", "TEXT", 75),
    ("CODE", "DOUBLE", None),
    ("STAND", "TEXT", 10),
    ("CARRIER_OFFICIAL_NAME", "TEXT", 200),
    ("CARRIER_ALT_NAME", "TEXT", 200),
    ("FLAG_5G", "DOUBLE", None),
    ("FLAG_WHS", "DOUBLE", None),
    ("FLAG_GK", "DOUBLE", None)
]


existing_fields = {
    field.name.upper()
    for field in arcpy.ListFields(fc)
}


for name, field_type, length in fields_to_add:

    if name.upper() in existing_fields:

        print("Feld bereits vorhanden:", name)
        continue

    if field_type == "TEXT":

        arcpy.management.AddField(
            in_table=fc,
            field_name=name,
            field_type=field_type,
            field_length=length
        )

    else:

        arcpy.management.AddField(
            in_table=fc,
            field_name=name,
            field_type=field_type
        )

    print("Feld angelegt:", name)


print("\nFeldvorbereitung abgeschlossen.")

Vorhandene Felder:

OBJECTID                       OID          required=True
Shape                          Geometry     required=True
Name                           String       required=False
FolderPath                     String       required=False
SymbolID                       Integer      required=False
AltMode                        SmallInteger required=False
Base                           Double       required=False
Clamped                        SmallInteger required=False
Extruded                       SmallInteger required=False
Snippet                        String       required=False
PopupInfo                      String       required=False
Shape_Length                   Double       required=True

Zu löschende Felder: ['SymbolID', 'AltMode', 'Base', 'Clamped', 'Extruded', 'Snippet', 'PopupInfo']
Feld angelegt: LR_NR
Feld angelegt: LR_TYP
Feld angelegt: LR_PKC
Feld angelegt: LR_PEC
Feld angelegt: CODE
Feld angelegt: STAND
Feld angelegt: CARRIER_OFFICIAL_NAME
Feld ange

In [ ]:
#Schritt 8 Projizierte Daten zur Karte hinzufügen

In [12]:
# ============================================================
# PROJIZIERTE DATEN ZUR KARTE HINZUFÜGEN
# ============================================================

existing_map_layers = [
    layer.name
    for layer in active_map.listLayers()
]


if "polylinetofeature_Projected" not in existing_map_layers:

    active_map.addDataFromPath(
        PROJECTED_OUTPUT
    )

    print(
        "polylinetofeature_Projected wurde "
        "zur Karte hinzugefügt."
    )

else:

    print(
        "polylinetofeature_Projected ist bereits "
        "in der Karte vorhanden."
    )

polylinetofeature_Projected ist bereits in der Karte vorhanden.


In [ ]:
#Schritt 9 Dublettenprüfung durchführen

In [13]:
# ============================================================
# DUBLETTENPRÜFUNG NACH LINIENLÄNGE
# ============================================================

my_layer = PROJECTED_OUTPUT

UNIQUE_OUTPUT = os.path.join(
    OUTPUT_GDB,
    "Unique_FeatureClass"
)

DUPLICATED_OUTPUT = os.path.join(
    OUTPUT_GDB,
    "Duplicated_FeatureClass"
)


# Alte Ergebnisse löschen
for old_output in [
    UNIQUE_OUTPUT,
    DUPLICATED_OUTPUT
]:

    if arcpy.Exists(old_output):
        arcpy.management.Delete(old_output)


# Alte temporäre Layer löschen
for temporary_layer in [
    "Unique",
    "Duplicated"
]:

    if arcpy.Exists(temporary_layer):
        arcpy.management.Delete(temporary_layer)


# ------------------------------------------------------------
# Linienlängen und ObjectIDs einlesen
# ------------------------------------------------------------

records = []

with arcpy.da.SearchCursor(
    my_layer,
    ["SHAPE@", "OID@"]
) as cursor:

    for geometry, object_id in cursor:

        if geometry is None:
            continue

        records.append(
            [
                geometry.length,
                object_id
            ]
        )


print(
    "Anzahl der geprüften Features:",
    len(records)
)


# ------------------------------------------------------------
# Doppelte Linienlängen ermitteln
# ------------------------------------------------------------

tracked_lengths = []
duplicate_ids = []
unique_ids = []

for length, object_id in records:

    if length in tracked_lengths:
        duplicate_ids.append(object_id)

    else:
        tracked_lengths.append(length)
        unique_ids.append(object_id)


print("Gefundene Dubletten:", len(duplicate_ids))
print("Eindeutige Features:", len(unique_ids))


# ------------------------------------------------------------
# SQL-Abfragen erstellen
# ------------------------------------------------------------

oid_field = arcpy.Describe(
    my_layer
).OIDFieldName

oid_field_sql = arcpy.AddFieldDelimiters(
    my_layer,
    oid_field
)


if duplicate_ids:

    duplicate_id_text = ",".join(
        str(object_id)
        for object_id in duplicate_ids
    )

    duplicated_query = (
        f"{oid_field_sql} IN "
        f"({duplicate_id_text})"
    )

    unique_query = (
        f"{oid_field_sql} NOT IN "
        f"({duplicate_id_text})"
    )

    arcpy.management.MakeFeatureLayer(
        in_features=my_layer,
        out_layer="Duplicated",
        where_clause=duplicated_query
    )

    arcpy.management.MakeFeatureLayer(
        in_features=my_layer,
        out_layer="Unique",
        where_clause=unique_query
    )

else:

    # Leerer Dubletten-Layer
    arcpy.management.MakeFeatureLayer(
        in_features=my_layer,
        out_layer="Duplicated",
        where_clause="1 = 0"
    )

    # Alle Features sind eindeutig
    arcpy.management.MakeFeatureLayer(
        in_features=my_layer,
        out_layer="Unique"
    )


# ------------------------------------------------------------
# Ergebnisse speichern
# ------------------------------------------------------------

arcpy.management.CopyFeatures(
    in_features="Duplicated",
    out_feature_class=DUPLICATED_OUTPUT
)

arcpy.management.CopyFeatures(
    in_features="Unique",
    out_feature_class=UNIQUE_OUTPUT
)


duplicate_count = int(
    arcpy.management.GetCount(
        DUPLICATED_OUTPUT
    )[0]
)

unique_count = int(
    arcpy.management.GetCount(
        UNIQUE_OUTPUT
    )[0]
)


print("\nDubletten:", DUPLICATED_OUTPUT)
print("Anzahl der Dubletten:", duplicate_count)

print("\nEindeutige Features:", UNIQUE_OUTPUT)
print("Anzahl eindeutiger Features:", unique_count)

Anzahl der geprüften Features: 53761
Gefundene Dubletten: 1470
Eindeutige Features: 52291

Dubletten: Q:\SNAP\SNAP_GIS_Analyse\Adam_Analyse\\FTDB_Bearbeitung\BSNetze\BSNetze.gdb\Duplicated_FeatureClass
Anzahl der Dubletten: 1470

Eindeutige Features: Q:\SNAP\SNAP_GIS_Analyse\Adam_Analyse\\FTDB_Bearbeitung\BSNetze\BSNetze.gdb\Unique_FeatureClass
Anzahl eindeutiger Features: 52299


In [ ]:
#Schritt 10 Ergebnisse zur Karte hinzufügen

In [17]:
# ============================================================
# UNIQUE- UND DUBLETTEN-LAYER ZUR KARTE HINZUFÜGEN
# ============================================================

current_layer_names = [
    layer.name
    for layer in active_map.listLayers()
]


if "Unique_FeatureClass" not in current_layer_names:

    active_map.addDataFromPath(
        UNIQUE_OUTPUT
    )

    print(
        "Unique_FeatureClass wurde "
        "zur Karte hinzugefügt."
    )


if "Duplicated_FeatureClass" not in current_layer_names:

    active_map.addDataFromPath(
        DUPLICATED_OUTPUT
    )

    print(
        "Duplicated_FeatureClass wurde "
        "zur Karte hinzugefügt."
    )

In [ ]:
#Schritt 11 Daten in die Fremdtrassen-DB übertragen

In [ ]:
# ============================================================
# DATEN AN DIE FREMDTRASSEN-DATENBANK ANHÄNGEN
# ============================================================

TARGET_FEATURE_CLASS = (
    r"E:\Pfad\zur\Datenbankverbindung.sde"
    r"\OWNER.EXTERNE_FREMDTRASSEN"
)


if not TARGET_FEATURE_CLASS:
    raise ValueError(
        "TARGET_FEATURE_CLASS wurde nicht eingetragen."
    )


if not arcpy.Exists(TARGET_FEATURE_CLASS):
    raise RuntimeError(
        f"Die Ziel-Feature-Class wurde nicht gefunden:\n"
        f"{TARGET_FEATURE_CLASS}"
    )


arcpy.management.Append(
    inputs=UNIQUE_OUTPUT,
    target=TARGET_FEATURE_CLASS,
    schema_type="NO_TEST"
)


print(
    "Unique_FeatureClass wurde erfolgreich "
    "an die Zieldatenbank angehängt."
)

In [ ]:
#Schritt 12 – Optional: Temporäre Daten löschen

In [ ]:
# ============================================================
# TEMPORÄRE DATEN LÖSCHEN
# NUR GANZ AM ENDE AUSFÜHREN
# ============================================================

datasets_to_delete = [
    UNIQUE_OUTPUT,
    DUPLICATED_OUTPUT,
    POLYLINE_OUTPUT,
    PROJECTED_OUTPUT,
    KML_GDB,
    KML_LAYER_FILE
]


for dataset in datasets_to_delete:

    if arcpy.Exists(dataset):

        arcpy.management.Delete(dataset)
        print("Gelöscht:", dataset)


print("Bereinigung abgeschlossen.")

In [ ]:
++++++++++++++++++++++++++++++++++++++++Version_2+++++++++++++++++++++++++++++++++++++++

In [ ]:
#Bibliotheken und Pfade

In [3]:
# ============================================================
# ZELLE 1 – BIBLIOTHEKEN, PFADE UND KML-LAYER
# ============================================================

import arcpy
import os

arcpy.env.overwriteOutput = True


# ============================================================
# BENUTZEREINSTELLUNGEN
# ============================================================

# Name des obersten KML-Layers im Bereich "Contents"
kml_layer_name = "~GEs14316"


# Ausgabeordner
# Den Anfang des Pfades vollständig eintragen
out_put = (
    r"Q:\SNAP\SNAP_GIS_Analyse\Adam_Analyse"
    r"\FTDB_Bearbeitung"
    r"\Erste_10Carrier"
    
   
    
)


# Name der Arbeits-Geodatabase
output_gdb_name = "Erste_10Carrier.gdb"


# Vollständiger Pfad zur Arbeits-Geodatabase
output_gdb = os.path.join(
    out_put,
    output_gdb_name
)


# Name der KML-Konvertierung
kml_conversion_name = "kmltolayer"


# Ziel-Feature-Class in der Fremdtrassen-Datenbank
# Wird erst später benötigt
target_feature_class = r""


# ============================================================
# AUSGABEORDNER PRÜFEN
# ============================================================

print("Ausgabeordner:", out_put)
print(
    "Ausgabeordner vorhanden:",
    os.path.isdir(out_put)
)


if not os.path.isdir(out_put):
    raise RuntimeError(
        f"Der Ausgabeordner wurde nicht gefunden:\n"
        f"{out_put}\n\n"
        f"Bitte den Pfad in 'out_put' korrigieren."
    )


# ============================================================
# ARBEITS-GEODATABASE ANLEGEN ODER VERWENDEN
# ============================================================

if not arcpy.Exists(output_gdb):

    arcpy.management.CreateFileGDB(
        out_folder_path=out_put,
        out_name=output_gdb_name
    )

    print(
        "Arbeits-Geodatabase wurde erstellt:",
        output_gdb
    )

else:

    print(
        "Arbeits-Geodatabase ist bereits vorhanden:",
        output_gdb
    )


# Workspace setzen
arcpy.env.workspace = output_gdb

print("Workspace wurde gesetzt:", arcpy.env.workspace)


# ============================================================
# KML-LAYER IN DER AKTIVEN KARTE SUCHEN
# ============================================================

project = arcpy.mp.ArcGISProject("CURRENT")
active_map = project.activeMap


if active_map is None:
    raise RuntimeError(
        "In ArcGIS Pro ist keine aktive Karte geöffnet."
    )


matching_layers = [
    layer
    for layer in active_map.listLayers()
    if layer.name == kml_layer_name
]


if not matching_layers:

    available_layers = [
        layer.name
        for layer in active_map.listLayers()
    ]

    raise RuntimeError(
        f"Der KML-Layer '{kml_layer_name}' wurde "
        f"nicht gefunden.\n"
        f"Vorhandene Layer: {available_layers}"
    )


# Gefundener ArcGIS-KML-Layer
input_kml = matching_layers[0]


# ============================================================
# EINSTELLUNGEN ANZEIGEN
# ============================================================

print("\nEinstellungen wurden erfolgreich geladen.")
print("KML-Layer:", input_kml.name)
print("Ausgabeordner:", out_put)
print("Arbeits-Geodatabase:", output_gdb)
print("KML-Konvertierungsname:", kml_conversion_name)

Ausgabeordner: Q:\SNAP\SNAP_GIS_Analyse\Adam_Analyse\FTDB_Bearbeitung\Erste_10Carrier
Ausgabeordner vorhanden: True
Arbeits-Geodatabase wurde erstellt: Q:\SNAP\SNAP_GIS_Analyse\Adam_Analyse\FTDB_Bearbeitung\Erste_10Carrier\Erste_10Carrier.gdb
Workspace wurde gesetzt: Q:\SNAP\SNAP_GIS_Analyse\Adam_Analyse\FTDB_Bearbeitung\Erste_10Carrier\Erste_10Carrier.gdb


RuntimeError: Der KML-Layer '~GEs14316' wurde nicht gefunden.
Vorhandene Layer: ['Dokom', 'Unique_FeatureClass_Dokom', 'DNSNet', 'Unique_FeatureClass_DNSNet_Gesammt', 'Unique_FeatureClass_DNSNet26', 'Unique_FeatureClass_DNSNet', 'Deutsche_Glasfaser', 'Unique_FeatureClass_Deutsche_Glasfaser', 'Unique_FeatureClass_NrwSuedBundeslaender', 'Unique_FeatureClass_NrwWarendorf', 'Unique_FeatureClass_NrwSudwest', 'Unique_FeatureClass_Nrwost', 'Unique_FeatureClass_NrwNordwest', 'Unique_FeatureClass_NrwNord', 'Unique_FeatureClass_NoerdlicheBundeslaender', 'Deutsche_Bahn', 'Unique_FeatureClass_DeutscheBahn', 'DBBroadband', 'Unique_FeatureClass', 'BSNetze', 'BSNetze_Featur_Class', 'Colt', 'Unique_FeatureClass_Colt', 'Unique_FeatureClass_Stuttgart', 'Unique_FeatureClass_Munich', 'Unique_FeatureClass_Hannover', 'Unique_FeatureClass_Hamburg', 'Unique_FeatureClass_Frankfurt2', 'Unique_FeatureClass_Dusseldorf', 'Unique_FeatureClass_Cologne', 'Unique_FeatureClass_Berlin', 'Unique_FeatureClass_Frankfurt', 

In [ ]:
#ZELLE 2 KML in Layer konvertieren

In [2]:
# ============================================================
# ZELLE 2 – KML/KMZ IN EINE KML-GEODATABASE KONVERTIEREN
# ============================================================

# KMLToLayer erzeugt eine separate Geodatabase:
# kmltolayer.gdb
kml_gdb = os.path.join(
    out_put,
    kml_conversion_name + ".gdb"
)


# Zusätzlich wird eine Layer-Datei erzeugt:
# kmltolayer.lyrx
kml_layer_file = os.path.join(
    out_put,
    kml_conversion_name + ".lyrx"
)


print("Arbeits-Geodatabase:", output_gdb)
print("KML-Geodatabase:", kml_gdb)
print("KML-Layer-Datei:", kml_layer_file)


# ============================================================
# ALTE KML-KONVERTIERUNG LÖSCHEN
# BSNetze.gdb WIRD NICHT GELÖSCHT
# ============================================================

for old_output in [
    kml_gdb,
    kml_layer_file
]:

    if arcpy.Exists(old_output):

        arcpy.management.Delete(old_output)

        print(
            "Alte KML-Ausgabe gelöscht:",
            old_output
        )


# ============================================================
# KML-LAYER KONVERTIEREN
# ============================================================

print("KML-Konvertierung wird gestartet ...")


arcpy.conversion.KMLToLayer(
    in_kml_file=input_kml,
    output_folder=out_put,
    output_data=kml_conversion_name
)


# ============================================================
# ERGEBNIS PRÜFEN
# ============================================================

print("KML-Konvertierung abgeschlossen.")
print("Erzeugte KML-Geodatabase:", kml_gdb)
print(
    "KML-Geodatabase vorhanden:",
    arcpy.Exists(kml_gdb)
)

print("Arbeits-Geodatabase:", output_gdb)
print(
    "Arbeits-Geodatabase vorhanden:",
    arcpy.Exists(output_gdb)
)

Arbeits-Geodatabase: Q:\SNAP\SNAP_GIS_Analyse\Adam_Analyse\FTDB_Bearbeitung\Dokom\Dokom.gdb
KML-Geodatabase: Q:\SNAP\SNAP_GIS_Analyse\Adam_Analyse\FTDB_Bearbeitung\Dokom\kmltolayer.gdb
KML-Layer-Datei: Q:\SNAP\SNAP_GIS_Analyse\Adam_Analyse\FTDB_Bearbeitung\Dokom\kmltolayer.lyrx
KML-Konvertierung wird gestartet ...
KML-Konvertierung abgeschlossen.
Erzeugte KML-Geodatabase: Q:\SNAP\SNAP_GIS_Analyse\Adam_Analyse\FTDB_Bearbeitung\Dokom\kmltolayer.gdb
KML-Geodatabase vorhanden: True
Arbeits-Geodatabase: Q:\SNAP\SNAP_GIS_Analyse\Adam_Analyse\FTDB_Bearbeitung\Dokom\Dokom.gdb
Arbeits-Geodatabase vorhanden: True


In [ ]:
#Zelle 3 – Polylinien exportieren

In [3]:
# ============================================================
# ZELLE 3 – POLYLINIEN IN DIE ARBEITS-GEODATABASE EXPORTIEREN
# ============================================================

polyline_fc = os.path.join(
    kml_gdb,
    "Placemarks",
    "Polylines"
)

polyline_output = os.path.join(
    output_gdb,
    "polylinetofeature"
)


print("Polyline-Quelle:", polyline_fc)
print("Polyline-Quelle vorhanden:", arcpy.Exists(polyline_fc))


if not arcpy.Exists(polyline_fc):
    raise RuntimeError(
        "Die Feature-Class 'Placemarks\\Polylines' "
        "wurde nicht gefunden."
    )


# Altes Ergebnis löschen
if arcpy.Exists(polyline_output):
    arcpy.management.Delete(polyline_output)


# Polylinien exportieren
arcpy.conversion.FeatureClassToFeatureClass(
    polyline_fc,
    output_gdb,
    "polylinetofeature",
    "",
    "",
    ""
)


polyline_count = int(
    arcpy.management.GetCount(polyline_output)[0]
)


print("Polylinien wurden exportiert.")
print("Ausgabe:", polyline_output)
print("Anzahl der Polylinien:", polyline_count)

Polyline-Quelle: Q:\SNAP\SNAP_GIS_Analyse\Adam_Analyse\FTDB_Bearbeitung\Dokom\kmltolayer.gdb\Placemarks\Polylines
Polyline-Quelle vorhanden: True
Polylinien wurden exportiert.
Ausgabe: Q:\SNAP\SNAP_GIS_Analyse\Adam_Analyse\FTDB_Bearbeitung\Dokom\Dokom.gdb\polylinetofeature
Anzahl der Polylinien: 5050


In [ ]:
#Zelle 4 – Nach ETRS89 / UTM Zone 32N projizieren

In [4]:
# ============================================================
# ZELLE 4 – PROJEKTION NACH ETRS89 / UTM ZONE 32N
# EPSG:25832
# ============================================================

projected_output = os.path.join(
    output_gdb,
    "polylinetofeature_Projected"
)


# Altes Ergebnis löschen
if arcpy.Exists(projected_output):
    arcpy.management.Delete(projected_output)


arcpy.management.Project(
    "polylinetofeature",
    projected_output,
    'PROJCS["ETRS_1989_UTM_Zone_32N",'
    'GEOGCS["GCS_ETRS_1989",'
    'DATUM["D_ETRS_1989",'
    'SPHEROID["GRS_1980",6378137.0,298.257222101]],'
    'PRIMEM["Greenwich",0.0],'
    'UNIT["Degree",0.0174532925199433]],'
    'PROJECTION["Transverse_Mercator"],'
    'PARAMETER["False_Easting",500000.0],'
    'PARAMETER["False_Northing",0.0],'
    'PARAMETER["Central_Meridian",9.0],'
    'PARAMETER["Scale_Factor",0.9996],'
    'PARAMETER["Latitude_Of_Origin",0.0],'
    'UNIT["Meter",1.0]]',
    "ETRS_1989_To_WGS_1984",
    None,
    "NO_PRESERVE_SHAPE",
    None,
    "NO_VERTICAL"
)


projected_count = int(
    arcpy.management.GetCount(projected_output)[0]
)


print("Projektion abgeschlossen.")
print("Ausgabe:", projected_output)
print("Anzahl der Features:", projected_count)

Projektion abgeschlossen.
Ausgabe: Q:\SNAP\SNAP_GIS_Analyse\Adam_Analyse\FTDB_Bearbeitung\Dokom\Dokom.gdb\polylinetofeature_Projected
Anzahl der Features: 5050


In [ ]:
#Zelle 5 – Vorhandene Felder anzeigen

In [5]:
# ============================================================
# ZELLE 5 – FELDER PRÜFEN UND NAME UMBENENNEN
# ============================================================

fc = projected_output


# ============================================================
# FEATURE-CLASS PRÜFEN
# ============================================================

print("Feature-Class:", fc)
print("Feature-Class vorhanden:", arcpy.Exists(fc))


if not arcpy.Exists(fc):
    raise RuntimeError(
        f"Projizierte Feature-Class wurde nicht gefunden:\n"
        f"{fc}"
    )


# ============================================================
# VORHANDENE FELDNAMEN ERMITTELN
# ============================================================

existing_fields = {
    field.name.lower(): field.name
    for field in arcpy.ListFields(fc)
}


# ============================================================
# FELD FolderPath PRÜFEN
# ============================================================

if "folderpath" not in existing_fields:

    raise RuntimeError(
        "Das Feld 'FolderPath' wurde nicht gefunden."
    )

else:

    print(
        "Feld gefunden:",
        existing_fields["folderpath"]
    )


# ============================================================
# FELD Name IN TRASSEBESCHREIBUNG UMBENENNEN
# ============================================================

if "name" in existing_fields:

    original_name_field = existing_fields["name"]

    arcpy.management.AlterField(
        in_table=fc,
        field=original_name_field,
        new_field_name="TRASSEBESCHREIBUNG",
        new_field_alias="TRASSEBESCHREIBUNG"
    )

    print(
        f"Feld '{original_name_field}' wurde in "
        f"'TRASSEBESCHREIBUNG' umbenannt."
    )


elif "trassebeschreibung" in existing_fields:

    print(
        "Das Feld wurde bereits in "
        "'TRASSEBESCHREIBUNG' umbenannt."
    )


else:

    raise RuntimeError(
        "Das Feld 'Name' wurde nicht gefunden."
    )


# ============================================================
# FELDER NACH DER ÄNDERUNG ANZEIGEN
# ============================================================

print("\nFelder nach der Umbenennung:\n")


for position, field in enumerate(
    arcpy.ListFields(fc),
    start=1
):

    print(
        f"{position:2}. "
        f"{field.name:30} "
        f"{field.type:12} "
        f"required={field.required}"
    )

Feature-Class: Q:\SNAP\SNAP_GIS_Analyse\Adam_Analyse\FTDB_Bearbeitung\Dokom\Dokom.gdb\polylinetofeature_Projected
Feature-Class vorhanden: True
Feld gefunden: FolderPath
Feld 'Name' wurde in 'TRASSEBESCHREIBUNG' umbenannt.

Felder nach der Umbenennung:

 1. OBJECTID                       OID          required=True
 2. Shape                          Geometry     required=True
 3. TRASSEBESCHREIBUNG             String       required=False
 4. FolderPath                     String       required=False
 5. SymbolID                       Integer      required=False
 6. AltMode                        SmallInteger required=False
 7. Base                           Double       required=False
 8. Clamped                        SmallInteger required=False
 9. Extruded                       SmallInteger required=False
10. Snippet                        String       required=False
11. PopupInfo                      String       required=False
12. Shape_Length                   Double       require

In [ ]:
#Zelle 6 – Nicht benötigte Felder löschen

In [6]:
# ============================================================
# ZELLE 6 – NICHT BENÖTIGTE FELDER LÖSCHEN
# ============================================================

keep = {
    "FolderPath",
    "TRASSEBESCHREIBUNG"
}

drop = []


for field in arcpy.ListFields(fc):

    if field.required:
        continue

    if field.name in keep:
        continue

    drop.append(field.name)


print("Zu löschende Felder:", drop)


if drop:
    arcpy.management.DeleteField(
        fc,
        drop
    )

    print("Nicht benötigte Felder wurden gelöscht.")

else:
    print("Es wurden keine Felder gelöscht.")

Zu löschende Felder: ['SymbolID', 'AltMode', 'Base', 'Clamped', 'Extruded', 'Snippet', 'PopupInfo']
Nicht benötigte Felder wurden gelöscht.


In [ ]:
#Zelle 7 – Neue Felder anlegen

In [7]:
# ============================================================
# ZELLE 7 – NEUE FELDER FÜR DIE FREMDTRASSEN-DB ANLEGEN
# ============================================================

fields_to_add = [
    ("LR_NR", "TEXT", 50),
    ("LR_TYP", "TEXT", 100),
    ("ROHR_TYP", "TEXT", 50),
    ("LR_PKC", "TEXT", 50),
    ("LR_PEC", "TEXT", 75),
    ("CODE", "DOUBLE", None),
    ("TIMESTAMP", "DATE", None),
    ("CARRIER_OFFICIAL_NAME", "TEXT", 200),
    ("CARRIER_ALT_NAME", "TEXT", 200),
    ("KREDITOR_ID_SCM", "TEXT", 50),
    ("LR_NR_CARRIER", "TEXT", 50),
    ("BAU_SATUS", "TEXT", 50),
    ("GUELTIG_VON", "DATE", None),
    ("GUELTIG_BIS", "DATE", None),
    ("AKTUELL", "SHORT", None),
    ("VERSION", "TEXT", 5),
    ("ZWECK", "TEXT", 20)
]


# Vorhandene Felder ermitteln
existing_fields = {
    field.name.upper()
    for field in arcpy.ListFields(fc)
}


# Fehlende Felder hinzufügen
for name, field_type, length in fields_to_add:

    if name.upper() in existing_fields:
        print("Feld bereits vorhanden:", name)
        continue

    if field_type == "TEXT":
        arcpy.management.AddField(
            fc,
            name,
            field_type,
            field_length=length
        )

    else:
        arcpy.management.AddField(
            fc,
            name,
            field_type
        )

    print("Feld angelegt:", name)


print("Alle neuen Felder wurden geprüft und angelegt.")

Feld angelegt: LR_NR
Feld angelegt: LR_TYP
Feld angelegt: ROHR_TYP
Feld angelegt: LR_PKC
Feld angelegt: LR_PEC
Feld angelegt: CODE
Feld angelegt: TIMESTAMP
Feld angelegt: CARRIER_OFFICIAL_NAME
Feld angelegt: CARRIER_ALT_NAME
Feld angelegt: KREDITOR_ID_SCM
Feld angelegt: LR_NR_CARRIER
Feld angelegt: BAU_SATUS
Feld angelegt: GUELTIG_VON
Feld angelegt: GUELTIG_BIS
Feld angelegt: AKTUELL
Feld angelegt: VERSION
Feld angelegt: ZWECK
Alle neuen Felder wurden geprüft und angelegt.


In [ ]:
#Zelle 8 – Projizierte Daten zur Karte hinzufügen

In [8]:
# ============================================================
# ZELLE 8 – ERGEBNIS ZUR AKTIVEN KARTE HINZUFÜGEN
# ============================================================

project = arcpy.mp.ArcGISProject("CURRENT")
active_map = project.activeMap


if active_map is None:
    raise RuntimeError(
        "In ArcGIS Pro ist keine aktive Karte geöffnet."
    )


existing_layer_names = [
    layer.name
    for layer in active_map.listLayers()
]


if "polylinetofeature_Projected" not in existing_layer_names:

    active_map.addDataFromPath(fc)

    print(
        "polylinetofeature_Projected wurde "
        "zur aktiven Karte hinzugefügt."
    )

else:

    print(
        "polylinetofeature_Projected ist bereits "
        "in der Karte vorhanden."
    )

polylinetofeature_Projected ist bereits in der Karte vorhanden.


In [ ]:
#Zelle 9 – Dubletten anhand der Linienlänge ermitteln

In [9]:
# ============================================================
# ZELLE 9 – DUBLETTENPRÜFUNG NACH LINIENLÄNGE
# ============================================================

my_layer = projected_output

Unique_ID = []
Duplicate_ID = []
my_set = []


with arcpy.da.SearchCursor(
    my_layer,
    ["SHAPE@", "OID@"]
) as cursor:

    for geometry, feature_id in cursor:

        if geometry is None:
            continue

        length = geometry.length

        my_set.append(
            [
                length,
                feature_id
            ]
        )


print(
    "Anzahl der Features in my_set:",
    len(my_set)
)


track_array = []


for record in my_set:

    length = record[0]
    feature_id = record[1]
    key = length

    if key in track_array:
        Duplicate_ID.append(feature_id)

    else:
        track_array.append(key)
        Unique_ID.append(feature_id)


print("Anzahl der Dubletten:", len(Duplicate_ID))
print("Anzahl eindeutiger Features:", len(Unique_ID))
print("Dubletten-IDs:", Duplicate_ID)

Anzahl der Features in my_set: 5050
Anzahl der Dubletten: 11
Anzahl eindeutiger Features: 5039
Dubletten-IDs: [2958, 2962, 3018, 4148, 4630, 4643, 4665, 4714, 4919, 4920, 4921]


In [ ]:
#Zelle 10 – Unique- und Dubletten-Feature-Classes erstellen

In [10]:
# ============================================================
# ZELLE 10 – UNIQUE- UND DUBLETTEN-DATEN SPEICHERN
# ============================================================

duplicated_output = os.path.join(
    output_gdb,
    "Duplicated_FeatureClass"
)

unique_output = os.path.join(
    output_gdb,
    "Unique_FeatureClass"
)


# Alte Ausgaben löschen
for old_output in [
    duplicated_output,
    unique_output
]:

    if arcpy.Exists(old_output):
        arcpy.management.Delete(old_output)


# Temporäre Layer vorher löschen
for temporary_layer in [
    "Duplicated",
    "Unique"
]:

    if arcpy.Exists(temporary_layer):
        arcpy.management.Delete(temporary_layer)


# Tatsächlichen Namen des ObjectID-Feldes ermitteln
oid_field = arcpy.Describe(
    my_layer
).OIDFieldName

oid_field_sql = arcpy.AddFieldDelimiters(
    my_layer,
    oid_field
)


if Duplicate_ID:

    duplicate_text = ",".join(
        str(feature_id)
        for feature_id in Duplicate_ID
    )

    duplicated_query = (
        f"{oid_field_sql} IN ({duplicate_text})"
    )

    unique_query = (
        f"{oid_field_sql} NOT IN ({duplicate_text})"
    )


    arcpy.management.MakeFeatureLayer(
        my_layer,
        "Duplicated",
        duplicated_query
    )

    arcpy.management.MakeFeatureLayer(
        my_layer,
        "Unique",
        unique_query
    )

else:

    # Leerer Dubletten-Layer
    arcpy.management.MakeFeatureLayer(
        my_layer,
        "Duplicated",
        "1 = 0"
    )

    # Alle Datensätze als eindeutig behandeln
    arcpy.management.MakeFeatureLayer(
        my_layer,
        "Unique"
    )


arcpy.management.CopyFeatures(
    "Duplicated",
    duplicated_output
)


arcpy.management.CopyFeatures(
    "Unique",
    unique_output
)


duplicated_count = int(
    arcpy.management.GetCount(duplicated_output)[0]
)

unique_count = int(
    arcpy.management.GetCount(unique_output)[0]
)


print("Duplicated Feature-Class:", duplicated_output)
print("Anzahl Dubletten:", duplicated_count)

print("Unique Feature-Class:", unique_output)
print("Anzahl eindeutiger Features:", unique_count)

Duplicated Feature-Class: Q:\SNAP\SNAP_GIS_Analyse\Adam_Analyse\FTDB_Bearbeitung\Dokom\Dokom.gdb\Duplicated_FeatureClass
Anzahl Dubletten: 11
Unique Feature-Class: Q:\SNAP\SNAP_GIS_Analyse\Adam_Analyse\FTDB_Bearbeitung\Dokom\Dokom.gdb\Unique_FeatureClass
Anzahl eindeutiger Features: 5039


In [ ]:
#Zelle 11 – Unique- und Dubletten-Layer zur Karte hinzufügen

In [11]:
# ============================================================
# ZELLE 11 – ERGEBNISSE ZUR KARTE HINZUFÜGEN
# ============================================================

existing_layer_names = [
    layer.name
    for layer in active_map.listLayers()
]


if "Unique_FeatureClass" not in existing_layer_names:

    active_map.addDataFromPath(
        unique_output
    )

    print(
        "Unique_FeatureClass wurde "
        "zur Karte hinzugefügt."
    )


if "Duplicated_FeatureClass" not in existing_layer_names:

    active_map.addDataFromPath(
        duplicated_output
    )

    print(
        "Duplicated_FeatureClass wurde "
        "zur Karte hinzugefügt."
    )

In [ ]:
#Zelle 12 – Temporäre Daten löschen

In [12]:
# ============================================================
# ZELLE 13 – TEMPORÄRE DATEN LÖSCHEN
# UNIQUE_FEATURECLASS BLEIBT ERHALTEN
# ============================================================

layers_to_delete = [
    duplicated_output,
    polyline_output,
    projected_output,
    kml_gdb,
    kml_layer_file
]


for dataset in layers_to_delete:

    if arcpy.Exists(dataset):

        arcpy.management.Delete(dataset)
        print("Gelöscht:", dataset)


print("Bereinigung abgeschlossen.")
print("Unique_FeatureClass bleibt erhalten:", unique_output)

Gelöscht: Q:\SNAP\SNAP_GIS_Analyse\Adam_Analyse\FTDB_Bearbeitung\Dokom\Dokom.gdb\Duplicated_FeatureClass
Gelöscht: Q:\SNAP\SNAP_GIS_Analyse\Adam_Analyse\FTDB_Bearbeitung\Dokom\Dokom.gdb\polylinetofeature
Gelöscht: Q:\SNAP\SNAP_GIS_Analyse\Adam_Analyse\FTDB_Bearbeitung\Dokom\Dokom.gdb\polylinetofeature_Projected
Gelöscht: Q:\SNAP\SNAP_GIS_Analyse\Adam_Analyse\FTDB_Bearbeitung\Dokom\kmltolayer.gdb
Gelöscht: Q:\SNAP\SNAP_GIS_Analyse\Adam_Analyse\FTDB_Bearbeitung\Dokom\kmltolayer.lyrx
Bereinigung abgeschlossen.
Unique_FeatureClass bleibt erhalten: Q:\SNAP\SNAP_GIS_Analyse\Adam_Analyse\FTDB_Bearbeitung\Dokom\Dokom.gdb\Unique_FeatureClass


In [ ]:
#Zelle 13 – Optional: In die Fremdtrassen-Datenbank übertragen

In [ ]:
# ============================================================
# ZELLE 12 – OPTIONALER IMPORT IN DIE FREMDTRASSEN-DB
# ============================================================

target_feature_class = (
    r"E:\Pfad\zur\Datenbankverbindung.sde"
    r"\OWNER.EXTERNE_FREMDTRASSEN"
)


print("Quelle:", unique_output)
print("Ziel:", target_feature_class)


if not arcpy.Exists(unique_output):
    raise RuntimeError(
        "Unique_FeatureClass wurde nicht gefunden."
    )


if not arcpy.Exists(target_feature_class):
    raise RuntimeError(
        f"Ziel-Feature-Class wurde nicht gefunden:\n"
        f"{target_feature_class}"
    )


arcpy.management.Append(
    inputs=unique_output,
    target=target_feature_class,
    schema_type="NO_TEST"
)


print(
    "Die eindeutigen Features wurden erfolgreich "
    "in die Fremdtrassen-Datenbank übertragen."
)

In [13]:
# ============================================================
# FELDWERTE MIT FESTGELEGTEN DATENTYPEN EINTRAGEN
# ============================================================

import arcpy
from datetime import datetime


# Feature-Class aus den vorherigen Notebook-Zellen
#input_fc = fc

# Alternativ:
input_fc = r"Q:\SNAP\SNAP_GIS_Analyse\Adam_Analyse\FTDB_Bearbeitung\Dokom\Dokom.gdb\Unique_FeatureClass_Dokom"


# ============================================================
# 1. FELDDEFINITIONEN
# ============================================================

field_definitions = [
    ("LR_NR", "TEXT", 50),
    ("LR_TYP", "TEXT", 100),
    ("ROHR_TYP", "TEXT", 50),
    ("LR_PKC", "TEXT", 50),
    ("LR_PEC", "TEXT", 75),
    ("CODE", "DOUBLE", None),
    ("TIMESTAMP", "DATE", None),
    ("CARRIER_OFFICIAL_NAME", "TEXT", 200),
    ("CARRIER_ALT_NAME", "TEXT", 200),
    ("KREDITOR_ID_SCM", "TEXT", 50),
    ("LR_NR_CARRIER", "TEXT", 50),
    ("BAU_SATUS", "TEXT", 50),
    ("GUELTIG_VON", "DATE", None),
    ("GUELTIG_BIS", "DATE", None),
    ("AKTUELL", "SHORT", None),
    ("VERSION", "TEXT", 5),
    ("ZWECK", "TEXT", 20)
]


# ============================================================
# 2. HIER DIE WERTE EINTRAGEN
# ============================================================
#
# TEXT:   "Text"
# DOUBLE: 123.45
# DATE:   datetime(2026, 9, 9)
# SHORT:  1
# None:   Feld nicht verändern
# ============================================================

field_values = {
    "LR_NR": "0",
    "LR_TYP": "TRASSE",
    "ROHR_TYP": "0",
    "LR_PKC": "FREMD",
    "LR_PEC": "DOKOM",
    "CODE": 1,
    "TIMESTAMP": datetime(2024, 6, 18),
    "CARRIER_OFFICIAL_NAME": "DOKOM GmbH Dortmund",
    "CARRIER_ALT_NAME": "0",
    "KREDITOR_ID_SCM": "PCP0041",
    "LR_NR_CARRIER": "0",
    "BAU_SATUS": "0",
    "GUELTIG_VON": datetime(2024, 6, 18),

    # Fester Wert: 01.01.2050
    "GUELTIG_BIS": datetime(2050, 1, 1),

    "AKTUELL": 1,
    "VERSION": "v1",
    "ZWECK": "5G"
}


# ============================================================
# 3. FEATURE-CLASS PRÜFEN
# ============================================================

if not arcpy.Exists(input_fc):
    raise RuntimeError(
        f"Feature-Class wurde nicht gefunden:\n{input_fc}"
    )


existing_fields = {
    field.name.upper(): field
    for field in arcpy.ListFields(input_fc)
}


# ============================================================
# 4. WERTE UND DATENTYPEN PRÜFEN
# ============================================================

validated_values = {}


for field_name, expected_type, max_length in field_definitions:

    value = field_values.get(field_name)

    # None bedeutet: Feld überspringen
    if value is None:
        print(f"Übersprungen: {field_name}")
        continue

    field_key = field_name.upper()

    if field_key not in existing_fields:
        raise RuntimeError(
            f"Feld wurde nicht gefunden: {field_name}"
        )

    # --------------------------------------------------------
    # TEXT
    # --------------------------------------------------------

    if expected_type == "TEXT":

        if not isinstance(value, str):
            raise TypeError(
                f"{field_name} muss ein Textwert sein, "
                f"zum Beispiel \"Mein Text\"."
            )

        if max_length and len(value) > max_length:
            raise ValueError(
                f"Der Wert für {field_name} besitzt "
                f"{len(value)} Zeichen.\n"
                f"Erlaubt sind maximal {max_length} Zeichen."
            )

    # --------------------------------------------------------
    # DOUBLE
    # --------------------------------------------------------

    elif expected_type == "DOUBLE":

        if isinstance(value, bool) or not isinstance(
            value, (int, float)
        ):
            raise TypeError(
                f"{field_name} muss eine Zahl sein, "
                f"zum Beispiel 100.0."
            )

        value = float(value)

    # --------------------------------------------------------
    # DATE
    # --------------------------------------------------------

    elif expected_type == "DATE":

        if not isinstance(value, datetime):
            raise TypeError(
                f"{field_name} muss mit datetime angegeben werden, "
                f"zum Beispiel datetime(2026, 9, 9)."
            )

    # --------------------------------------------------------
    # SHORT
    # --------------------------------------------------------

    elif expected_type == "SHORT":

        if isinstance(value, bool) or not isinstance(value, int):
            raise TypeError(
                f"{field_name} muss eine ganze Zahl sein, "
                f"zum Beispiel 1."
            )

        # Wertebereich eines Short-Integer-Feldes
        if not -32768 <= value <= 32767:
            raise ValueError(
                f"{field_name} liegt außerhalb des zulässigen "
                f"SHORT-Wertebereichs."
            )

    validated_values[field_name] = value


if not validated_values:
    raise RuntimeError(
        "Es wurden keine Werte zum Eintragen angegeben."
    )


# ============================================================
# 5. VORSCHAU
# ============================================================

feature_count = int(
    arcpy.management.GetCount(input_fc)[0]
)


print("\nFeature-Class:", input_fc)
print("Anzahl der Features:", feature_count)
print("\nFolgende Werte werden eingetragen:")


for field_name, value in validated_values.items():
    print(f" -> {field_name}: {value!r}")


# ============================================================
# 6. ALLE FEATURES AKTUALISIEREN
# ============================================================

cursor_fields = list(validated_values.keys())
cursor_values = list(validated_values.values())

updated_count = 0


with arcpy.da.UpdateCursor(
    input_fc,
    cursor_fields
) as cursor:

    for row in cursor:

        for index, value in enumerate(cursor_values):
            row[index] = value

        cursor.updateRow(row)
        updated_count += 1


# ============================================================
# 7. ZUSAMMENFASSUNG
# ============================================================

print("\n========================================")
print("AKTUALISIERUNG ERFOLGREICH")
print("========================================")
print("Bearbeitete Features:", updated_count)
print("Ausgefüllte Felder:", len(cursor_fields))


Feature-Class: Q:\SNAP\SNAP_GIS_Analyse\Adam_Analyse\FTDB_Bearbeitung\Dokom\Dokom.gdb\Unique_FeatureClass_Dokom
Anzahl der Features: 5039

Folgende Werte werden eingetragen:
 -> LR_NR: '0'
 -> LR_TYP: 'TRASSE'
 -> ROHR_TYP: '0'
 -> LR_PKC: 'FREMD'
 -> LR_PEC: 'DOKOM'
 -> CODE: 1.0
 -> TIMESTAMP: datetime.datetime(2024, 6, 18, 0, 0)
 -> CARRIER_OFFICIAL_NAME: 'DOKOM GmbH Dortmund'
 -> CARRIER_ALT_NAME: '0'
 -> KREDITOR_ID_SCM: 'PCP0041'
 -> LR_NR_CARRIER: '0'
 -> BAU_SATUS: '0'
 -> GUELTIG_VON: datetime.datetime(2024, 6, 18, 0, 0)
 -> GUELTIG_BIS: datetime.datetime(2050, 1, 1, 0, 0)
 -> AKTUELL: 1
 -> VERSION: 'v1'
 -> ZWECK: '5G'

AKTUALISIERUNG ERFOLGREICH
Bearbeitete Features: 5039
Ausgefüllte Felder: 17


In [4]:
import arcpy
import os

# ==========================================
# 1. PARAMETER ANPASSEN
# ==========================================

# Zu mergende Polyline-Feature-Classes
input_fcs = [
    r"Q:\SNAP\SNAP_GIS_Analyse\Adam_Analyse\FTDB_Bearbeitung\Dokom\Dokom.gdb\Unique_FeatureClass_Dokom",  
    r"Q:\SNAP\SNAP_GIS_Analyse\Adam_Analyse\FTDB_Bearbeitung\DNSNet\DNSNet.gdb\Unique_FeatureClass_DNSNet_Gesammt",
    r"Q:\SNAP\SNAP_GIS_Analyse\Adam_Analyse\FTDB_Bearbeitung\Deutsche_Glasfaser\Deutsche_Glasfaser.gdb\Unique_FeatureClass_Deutsche_Glasfaser",
    r"Q:\SNAP\SNAP_GIS_Analyse\Adam_Analyse\FTDB_Bearbeitung\Deutsche_Bahn\Deutsche_Bahn.gdb\Unique_FeatureClass_DeutscheBahn",
    r"Q:\SNAP\SNAP_GIS_Analyse\Adam_Analyse\FTDB_Bearbeitung\DBBroadband\DBBroadband.gdb\Unique_FeatureClass",
    r"Q:\SNAP\SNAP_GIS_Analyse\Adam_Analyse\FTDB_Bearbeitung\BSNetze\BSNetze.gdb\BSNetze_Featur_Class",
    r"Q:\SNAP\SNAP_GIS_Analyse\Adam_Analyse\FTDB_Bearbeitung\Colt\Colt.gdb\Unique_FeatureClass_Colt",
    r"Q:\SNAP\SNAP_GIS_Analyse\Adam_Analyse\FTDB_Bearbeitung\FTDB_Bearbeitung.gdb\AVU_GEV_Reyhaneh",
    r"Q:\SNAP\SNAP_GIS_Analyse\Adam_Analyse\FTDB_Bearbeitung\FTDB_Bearbeitung.gdb\EDISCOM_Reyhaneh",
    r"Q:\SNAP\SNAP_GIS_Analyse\Adam_Analyse\FTDB_Bearbeitung\FTDB_Bearbeitung.gdb\EINSENERGY_Reyhaneh"
    
    
]

# Finale Polyline-Feature-Class
out_fc = (
    r"Q:\SNAP\SNAP_GIS_Analyse\Adam_Analyse"
    r"\FTDB_Bearbeitung\Erste_10Carrier\Erste_10Carrier.gdb"
    r"\FeatureClass_Erste_10Carrier"
)

# Vorhandene Ausgabe überschreiben
arcpy.env.overwriteOutput = True


# ==========================================
# 2. HAUPTFUNKTION
# ==========================================

def robust_merge_with_projection(
    input_features,
    output_feature_class
):
    print("=== 1. STARTE VORABPRÜFUNGEN ===")

    valid_fcs = []
    total_input_count = 0

    # ------------------------------------------
    # Eingabe-Feature-Classes prüfen
    # ------------------------------------------

    for fc in input_features:

        if not arcpy.Exists(fc):
            print(
                f"FEHLER: Feature-Class existiert nicht "
                f"und wird ignoriert:\n -> {fc}"
            )
            continue

        desc = arcpy.Describe(fc)
        shape_type = desc.shapeType
        count = int(arcpy.management.GetCount(fc)[0])

        print(
            f" -> '{desc.baseName}': "
            f"{count} Features | "
            f"Geometrie: {shape_type}"
        )

        # Nur Polylinien zulassen
        if shape_type.lower() != "polyline":
            print(
                f"    FEHLER: '{desc.baseName}' ist keine "
                f"Polyline-Feature-Class, sondern {shape_type}."
            )
            print("    -> Feature-Class wird ignoriert.")
            continue

        if count == 0:
            print(
                f"    WARNUNG: '{desc.baseName}' ist leer "
                f"und wird ignoriert."
            )
            continue

        valid_fcs.append(fc)
        total_input_count += count

    if len(valid_fcs) < 2:
        print(
            "\nABBRUCH: Es müssen mindestens zwei gültige, "
            "gefüllte Polyline-Feature-Classes vorhanden sein."
        )
        return

    # ------------------------------------------
    # Ziel-Koordinatensystem bestimmen
    # ------------------------------------------

    print(
        "\n=== 2. KOORDINATENSYSTEME PRÜFEN "
        "UND ANGLEICHEN ==="
    )

    # Erste Feature-Class mit bekanntem Koordinatensystem suchen
    target_sr = None

    for fc in valid_fcs:
        sr = arcpy.Describe(fc).spatialReference

        if sr and sr.name not in ("Unknown", ""):
            target_sr = sr
            break

    if target_sr is None:
        print(
            "ABBRUCH: Keine Eingabe besitzt ein bekanntes "
            "Koordinatensystem."
        )
        print(
            "Mindestens einer Feature-Class muss zuerst das "
            "richtige Koordinatensystem zugewiesen werden."
        )
        return

    print(f"Ziel-Koordinatensystem: {target_sr.name}")

    fcs_to_merge = []
    temp_fcs_to_delete = []

    try:

        for index, fc in enumerate(valid_fcs):

            desc = arcpy.Describe(fc)
            source_sr = desc.spatialReference

            # ------------------------------------------
            # Fall A: Unbekanntes Koordinatensystem
            # ------------------------------------------

            if (
                source_sr is None
                or source_sr.name in ("Unknown", "")
            ):
                print(
                    f" -> FEHLER: '{desc.baseName}' besitzt "
                    f"kein definiertes Koordinatensystem."
                )
                print(
                    "    Das Koordinatensystem kann nicht sicher "
                    "automatisch bestimmt werden."
                )
                print("    -> Feature-Class wird nicht gemergt.")
                continue

            # Koordinatensysteme vollständig vergleichen
            same_spatial_reference = (
                source_sr.exportToString()
                == target_sr.exportToString()
            )

            # ------------------------------------------
            # Fall B: Abweichendes Koordinatensystem
            # ------------------------------------------

            if not same_spatial_reference:

                print(
                    f" -> Projiziere '{desc.baseName}' von "
                    f"'{source_sr.name}' nach '{target_sr.name}'."
                )

                safe_name = arcpy.ValidateTableName(
                    f"proj_{index}_{desc.baseName}",
                    arcpy.env.scratchGDB
                )

                temp_fc_path = os.path.join(
                    arcpy.env.scratchGDB,
                    safe_name
                )

                if arcpy.Exists(temp_fc_path):
                    arcpy.management.Delete(temp_fc_path)

                # Geeignete geographische Transformation suchen
                transformations = arcpy.ListTransformations(
                    source_sr,
                    target_sr
                )

                transformation = (
                    transformations[0]
                    if transformations
                    else None
                )

                if transformation:
                    print(
                        "    Geographische Transformation:",
                        transformation
                    )

                    arcpy.management.Project(
                        in_dataset=fc,
                        out_dataset=temp_fc_path,
                        out_coor_system=target_sr,
                        transform_method=transformation
                    )

                else:
                    arcpy.management.Project(
                        in_dataset=fc,
                        out_dataset=temp_fc_path,
                        out_coor_system=target_sr
                    )

                fcs_to_merge.append(temp_fc_path)
                temp_fcs_to_delete.append(temp_fc_path)

            # ------------------------------------------
            # Fall C: Koordinatensystem stimmt überein
            # ------------------------------------------

            else:
                print(
                    f" -> '{desc.baseName}' besitzt bereits "
                    f"das Ziel-Koordinatensystem."
                )

                fcs_to_merge.append(fc)

        if len(fcs_to_merge) < 2:
            print(
                "\nABBRUCH: Nach der Koordinatensystemprüfung "
                "sind weniger als zwei Feature-Classes übrig."
            )
            return

        # Tatsächlich zu mergende Anzahl neu berechnen
        expected_count = sum(
            int(arcpy.management.GetCount(fc)[0])
            for fc in fcs_to_merge
        )

        # ------------------------------------------
        # Field Mappings
        # ------------------------------------------

        print("\n=== 3. FIELD MAPPINGS ERSTELLEN ===")
        print("Erstelle dynamische Feldzuordnungen ...")

        field_mappings = arcpy.FieldMappings()

        for fc in fcs_to_merge:
            field_mappings.addTable(fc)

        print(
            f" -> {field_mappings.fieldCount} "
            f"Ausgabefelder erkannt."
        )

        # ------------------------------------------
        # Vorhandene Ausgabe löschen
        # ------------------------------------------

        if arcpy.Exists(output_feature_class):
            print("\nVorhandene Ausgabe wird gelöscht:")
            print(f" -> {output_feature_class}")

            arcpy.management.Delete(output_feature_class)

        # ------------------------------------------
        # Merge
        # ------------------------------------------

        print("\n=== 4. POLYLINE-MERGE DURCHFÜHREN ===")
        print(
            f"Führe {len(fcs_to_merge)} "
            f"Polyline-Feature-Classes zusammen ..."
        )

        arcpy.management.Merge(
            inputs=fcs_to_merge,
            output=output_feature_class,
            field_mappings=field_mappings
        )

        # ------------------------------------------
        # Ergebnis kontrollieren
        # ------------------------------------------

        out_count = int(
            arcpy.management.GetCount(
                output_feature_class
            )[0]
        )

        out_desc = arcpy.Describe(output_feature_class)

        print("\n=== 5. ZUSAMMENFASSUNG ===")
        print("ERFOLG! Ausgabe:")
        print(f" -> {output_feature_class}")
        print(f"Geometrietyp:               {out_desc.shapeType}")
        print(f"Ziel-Koordinatensystem:     {target_sr.name}")
        print(f"Ursprünglich gefunden:      {total_input_count}")
        print(f"Zum Merge zugelassen:       {expected_count}")
        print(f"Tatsächliche Ausgabe:       {out_count}")

        if expected_count == out_count:
            print(
                "KONTROLLE ERFOLGREICH: "
                "Die Feature-Anzahlen stimmen überein."
            )
        else:
            print(
                "WARNUNG: Die Feature-Anzahl der Ausgabe "
                "entspricht nicht der Summe der verwendeten "
                "Eingaben."
            )

    except arcpy.ExecuteError:
        print("\nARCPY-FEHLER:")
        print(arcpy.GetMessages(2))

    except Exception as error:
        print("\nUNERWARTETER FEHLER:")
        print(error)

    finally:

        # ------------------------------------------
        # Temporäre projizierte Daten entfernen
        # ------------------------------------------

        if temp_fcs_to_delete:
            print(
                "\nRäume temporäre Feature-Classes "
                "aus der Scratch-GDB auf ..."
            )

            for temp_fc in temp_fcs_to_delete:

                if arcpy.Exists(temp_fc):

                    try:
                        arcpy.management.Delete(temp_fc)

                        print(
                            " -> Gelöscht:",
                            os.path.basename(temp_fc)
                        )

                    except Exception as delete_error:
                        print(
                            f" -> '{os.path.basename(temp_fc)}' "
                            f"konnte nicht gelöscht werden: "
                            f"{delete_error}"
                        )


# ==========================================
# 3. SKRIPT STARTEN
# ==========================================

robust_merge_with_projection(
    input_fcs,
    out_fc
)

=== 1. STARTE VORABPRÜFUNGEN ===
 -> 'Unique_FeatureClass_Dokom': 5039 Features | Geometrie: Polyline
 -> 'Unique_FeatureClass_DNSNet_Gesammt': 262535 Features | Geometrie: Polyline
 -> 'Unique_FeatureClass_Deutsche_Glasfaser': 593783 Features | Geometrie: Polyline
 -> 'Unique_FeatureClass_DeutscheBahn': 5219 Features | Geometrie: Polyline
 -> 'Unique_FeatureClass': 10309 Features | Geometrie: Polyline
 -> 'BSNetze_Featur_Class': 52299 Features | Geometrie: Polyline
 -> 'Unique_FeatureClass_Colt': 7446 Features | Geometrie: Polyline
 -> 'AVU_GEV_Reyhaneh': 652 Features | Geometrie: Polyline
 -> 'EDISCOM_Reyhaneh': 12129 Features | Geometrie: Polyline
 -> 'EINSENERGY_Reyhaneh': 4964 Features | Geometrie: Polyline

=== 2. KOORDINATENSYSTEME PRÜFEN UND ANGLEICHEN ===
Ziel-Koordinatensystem: ETRS_1989_UTM_Zone_32N
 -> 'Unique_FeatureClass_Dokom' besitzt bereits das Ziel-Koordinatensystem.
 -> 'Unique_FeatureClass_DNSNet_Gesammt' besitzt bereits das Ziel-Koordinatensystem.
 -> 'Unique_Featu

In [ ]:
#Dokom_Rohr_typ_Extraktion

In [2]:
# ============================================================
# ARCGIS-PRO-NOTEBOOK
# ROHRTYPEN AUS TRASSEBESCHREIBUNG EXTRAHIEREN
# ============================================================

import arcpy
import re


# ============================================================
# 1. EINSTELLUNGEN
# ============================================================

# Möglichkeit A:
# Variable aus den vorherigen Notebook-Zellen verwenden
#input_features = fc

# Möglichkeit B:
# Vollständigen Pfad direkt eintragen
input_features = (
    r"Q:\SNAP\SNAP_GIS_Analyse\Adam_Analyse\FTDB_Bearbeitung\Dokom\Dokom.gdb"
    r"\Unique_FeatureClass_Dokom"
)

source_field_name = "TRASSEBESCHREIBUNG"
target_field_name = "ROHR_TYP"


# True:
# DN100/40 wird als DN-100, DN-40 interpretiert
#
# False:
# Es wird nur DN-100 übernommen
extract_slash_diameters = True


# True:
# Wenn nichts erkannt wird, wird ROHR_TYP auf NULL gesetzt
#
# False:
# Ein vorhandener Wert in ROHR_TYP bleibt erhalten
clear_if_not_found = True


# Maximale Anzahl der ausgegebenen Beispiele
max_unrecognized_examples = 20
max_overflow_examples = 20
max_error_examples = 20


# ============================================================
# 2. REGULÄRE AUSDRÜCKE
# ============================================================

# Erkennt beispielsweise:
# DN 100
# DN100
# DN-100
# DN 100DN 40
# DN100/40
dn_pattern = re.compile(
    r"DN\s*[-–—]?\s*(\d{1,4})"
    r"(?:\s*/\s*(\d{1,4}))?",
    re.IGNORECASE
)


# Erkennt ausschließlich Werte mit expliziter Einheit mm:
# 40 mm
# 32mm
# 12,5 mm
mm_pattern = re.compile(
    r"(?<![\d.,])"
    r"(\d{1,4}(?:[.,]\d+)?)"
    r"\s*mm\b",
    re.IGNORECASE
)


# ============================================================
# 3. BEKANNTE ROHRBEZEICHNUNGEN
# ============================================================

# Aufbau:
# (
#     regulärer Ausdruck,
#     standardisierter Ausgabewert
# )
#
# Diese Liste kann später erweitert werden.

known_rohr_patterns = [
    (
        re.compile(
            r"\bFM\s*[-–—]\s*LWL\s*[-–—]\s*Leerrohr\b",
            re.IGNORECASE
        ),
        "FM-LWL-Leerrohr"
    ),

    # Beispiel für eine Erweiterung:
    #
    # (
    #     re.compile(
    #         r"\bKabelschutzrohr\b",
    #         re.IGNORECASE
    #     ),
    #     "Kabelschutzrohr"
    # ),
]


# ============================================================
# 4. HILFSFUNKTIONEN
# ============================================================

def normalize_dn(number_text):
    """
    Standardisiert DN-Angaben.

    Beispiel:
    100 -> DN-100
    040 -> DN-40
    """

    try:
        number = int(number_text)
        return f"DN-{number}"

    except (TypeError, ValueError):
        return None


def normalize_mm(number_text):
    """
    Standardisiert Millimeterangaben.

    Beispiel:
    40   -> 40 mm
    12,5 -> 12,5 mm
    """

    if number_text is None:
        return None

    value = str(number_text).strip()

    # Führende Nullen bei ganzen Zahlen entfernen
    if re.fullmatch(r"\d+", value):
        value = str(int(value))

    return f"{value} mm"


def spans_overlap(first_span, second_span):
    """
    Prüft, ob sich zwei Fundstellen im Text überschneiden.
    """

    first_start, first_end = first_span
    second_start, second_end = second_span

    return (
        first_start < second_end
        and second_start < first_end
    )


def remove_duplicates_preserve_order(values):
    """
    Entfernt doppelte Rohrtypen und behält die Reihenfolge.
    """

    result = []
    seen = set()

    for value in values:

        key = value.casefold()

        if key in seen:
            continue

        seen.add(key)
        result.append(value)

    return result


# ============================================================
# 5. EXTRAKTIONSFUNKTION
# ============================================================

def extract_rohr_typ(text):
    """
    Extrahiert und standardisiert Rohrtypen.

    Rückgabe:
        Liste unterschiedlicher Rohrtypen

    Beispiel:
        DN 100DN 100DN 40 sw/orDN 40 sw/ws

    Ergebnis:
        ["DN-100", "DN-40"]
    """

    if text is None:
        return []

    text = str(text).strip()

    if not text:
        return []

    # Aufbau:
    # Textposition, laufende Nummer, Ausgabewert
    candidates = []

    # Fundstellen der DN-Ausdrücke
    dn_spans = []

    sequence = 0

    # --------------------------------------------------------
    # A. DN-Angaben erkennen
    # --------------------------------------------------------

    for match in dn_pattern.finditer(text):

        first_dn = normalize_dn(
            match.group(1)
        )

        if first_dn:

            candidates.append(
                (
                    match.start(1),
                    sequence,
                    first_dn
                )
            )

            sequence += 1

        # Gesamten DN-Ausdruck speichern
        dn_spans.append(
            match.span()
        )

        # Zweiten Durchmesser aus DN100/40 übernehmen
        if (
            extract_slash_diameters
            and match.group(2) is not None
        ):

            second_dn = normalize_dn(
                match.group(2)
            )

            if second_dn:

                candidates.append(
                    (
                        match.start(2),
                        sequence,
                        second_dn
                    )
                )

                sequence += 1

    # --------------------------------------------------------
    # B. Millimeterangaben erkennen
    # --------------------------------------------------------

    for match in mm_pattern.finditer(text):

        # Beispiel "DN 100 mm":
        # Nicht gleichzeitig DN-100 und 100 mm erzeugen
        overlaps_dn = any(
            spans_overlap(
                match.span(),
                dn_span
            )
            for dn_span in dn_spans
        )

        if overlaps_dn:
            continue

        millimeter_value = normalize_mm(
            match.group(1)
        )

        if millimeter_value:

            candidates.append(
                (
                    match.start(),
                    sequence,
                    millimeter_value
                )
            )

            sequence += 1

    # --------------------------------------------------------
    # C. Bekannte Rohrbezeichnungen erkennen
    # --------------------------------------------------------

    for pattern, standard_value in known_rohr_patterns:

        for match in pattern.finditer(text):

            candidates.append(
                (
                    match.start(),
                    sequence,
                    standard_value
                )
            )

            sequence += 1

    # Reihenfolge aus der Beschreibung beibehalten
    candidates.sort(
        key=lambda item: (
            item[0],
            item[1]
        )
    )

    extracted_values = [
        item[2]
        for item in candidates
    ]

    return remove_duplicates_preserve_order(
        extracted_values
    )


# ============================================================
# 6. FEATURE-CLASS UND FELDER PRÜFEN
# ============================================================

if not arcpy.Exists(input_features):

    raise RuntimeError(
        "Die Feature-Class wurde nicht gefunden:\n"
        f"{input_features}"
    )


fields = {
    field.name.upper(): field
    for field in arcpy.ListFields(input_features)
}


if source_field_name.upper() not in fields:

    raise RuntimeError(
        f"Das Feld '{source_field_name}' "
        f"wurde nicht gefunden."
    )


if target_field_name.upper() not in fields:

    raise RuntimeError(
        f"Das Feld '{target_field_name}' "
        f"wurde nicht gefunden."
    )


source_field = fields[
    source_field_name.upper()
]

target_field = fields[
    target_field_name.upper()
]


if source_field.type != "String":

    raise RuntimeError(
        f"Das Feld '{source_field.name}' "
        f"ist kein Textfeld."
    )


if target_field.type != "String":

    raise RuntimeError(
        f"Das Feld '{target_field.name}' "
        f"ist kein Textfeld."
    )


target_length = target_field.length


if not target_length or target_length <= 0:

    raise RuntimeError(
        f"Die Länge von '{target_field.name}' "
        f"konnte nicht ermittelt werden."
    )


print("Feature-Class:", input_features)
print("Quellfeld:", source_field.name)
print("Zielfeld:", target_field.name)
print("Länge des Zielfeldes:", target_length)
print(
    "DN100/40-Regel:",
    "aktiviert"
    if extract_slash_diameters
    else "deaktiviert"
)


# ============================================================
# 7. STATISTIK VORBEREITEN
# ============================================================

total_count = 0
recognized_count = 0
stored_count = 0
not_recognized_count = 0
multiple_count = 0
overflow_count = 0
error_count = 0

all_recognized_types = set()

unrecognized_examples = []
unrecognized_seen = set()

overflow_examples = []
error_examples = []


cursor_fields = [
    "OID@",
    source_field.name,
    target_field.name
]


# ============================================================
# 8. ROHR_TYP AKTUALISIEREN
# ============================================================

print("\nExtraktion wird gestartet ...")


with arcpy.da.UpdateCursor(
    input_features,
    cursor_fields
) as cursor:

    for row in cursor:

        oid = row[0]
        description = row[1]

        total_count += 1

        try:

            extracted_types = extract_rohr_typ(
                description
            )

            # ------------------------------------------------
            # Kein Rohrtyp erkannt
            # ------------------------------------------------

            if not extracted_types:

                not_recognized_count += 1

                if (
                    description is None
                    or not str(description).strip()
                ):
                    example_text = "<NULL oder leer>"

                else:
                    example_text = str(
                        description
                    ).strip()

                if (
                    example_text not in unrecognized_seen
                    and len(unrecognized_examples)
                    < max_unrecognized_examples
                ):

                    unrecognized_seen.add(
                        example_text
                    )

                    unrecognized_examples.append(
                        (
                            oid,
                            example_text
                        )
                    )

                if clear_if_not_found:

                    # ROHR_TYP auf NULL setzen
                    row[2] = None
                    cursor.updateRow(row)

                continue

            # ------------------------------------------------
            # Rohrtyp erkannt
            # ------------------------------------------------

            recognized_count += 1

            if len(extracted_types) > 1:
                multiple_count += 1

            all_recognized_types.update(
                extracted_types
            )

            result = ", ".join(
                extracted_types
            )

            # ------------------------------------------------
            # Feldlänge kontrollieren
            # ------------------------------------------------

            if len(result) > target_length:

                overflow_count += 1

                # Vorhandenen Wert nicht überschreiben
                if (
                    len(overflow_examples)
                    < max_overflow_examples
                ):

                    overflow_examples.append(
                        (
                            oid,
                            description,
                            result,
                            len(result)
                        )
                    )

                continue

            # Ausschließlich ROHR_TYP ändern
            row[2] = result
            cursor.updateRow(row)

            stored_count += 1

        except Exception as row_error:

            error_count += 1

            if (
                len(error_examples)
                < max_error_examples
            ):

                error_examples.append(
                    (
                        oid,
                        str(row_error)
                    )
                )

        if total_count % 1000 == 0:

            print(
                f"{total_count} Datensätze verarbeitet ..."
            )


# ============================================================
# 9. QUALITÄTSKONTROLLE
# ============================================================

print("\n==========================================")
print("QUALITÄTSKONTROLLE")
print("==========================================")

print(
    "1. Untersuchte Datensätze:",
    total_count
)

print(
    "2. Datensätze mit erkanntem Rohrtyp:",
    recognized_count
)

print(
    "   Davon erfolgreich gespeichert:",
    stored_count
)

print(
    "3. Datensätze ohne erkannten Rohrtyp:",
    not_recognized_count
)

print(
    "4. Datensätze mit mehreren Rohrtypen:",
    multiple_count
)


# ============================================================
# 10. UNTERSCHIEDLICHE ROHRTYPEN
# ============================================================

print(
    "\n5. Alle unterschiedlichen "
    "erkannten Rohrtypen:"
)


if all_recognized_types:

    for rohr_typ in sorted(
        all_recognized_types,
        key=str.casefold
    ):

        print("   -", rohr_typ)

else:

    print("   Keine Rohrtypen erkannt.")


# ============================================================
# 11. NICHT ERKANNTE BESCHREIBUNGEN
# ============================================================

print(
    "\n6. Beispiele für nicht erkannte "
    "Beschreibungen:"
)


if unrecognized_examples:

    for oid, description in unrecognized_examples:

        print(
            f"   OID {oid}: {description}"
        )

else:

    print(
        "   Keine nicht erkannten Beschreibungen."
    )


# ============================================================
# 12. ZU LANGE ERGEBNISSE
# ============================================================

if overflow_count > 0:

    print("\n==========================================")
    print("WARNUNG: ZU LANGE ERGEBNISSE")
    print("==========================================")

    print(
        f"{overflow_count} Ergebnis(se) waren länger "
        f"als {target_length} Zeichen."
    )

    print(
        "Diese Ergebnisse wurden nicht abgeschnitten "
        "und nicht gespeichert."
    )

    print(
        "Die bisherigen ROHR_TYP-Werte dieser "
        "Datensätze blieben unverändert."
    )

    for (
        oid,
        description,
        result,
        result_length
    ) in overflow_examples:

        print(
            f"\nOID {oid}: "
            f"{result_length} Zeichen"
        )

        print(
            "Beschreibung:",
            description
        )

        print(
            "Erkanntes Ergebnis:",
            result
        )

    print(
        "\nSichere Lösung: Lege ein neues Textfeld "
        "'ROHR_TYP_LANG' mit einer Länge von 255 Zeichen "
        "an und speichere längere Ergebnisse dort."
    )


# ============================================================
# 13. FEHLER AUSGEBEN
# ============================================================

if error_count > 0:

    print("\n==========================================")
    print("FEHLER BEI EINZELNEN DATENSÄTZEN")
    print("==========================================")

    print(
        "Anzahl der Datensätze mit Fehler:",
        error_count
    )

    for oid, error_text in error_examples:

        print(
            f"OID {oid}: {error_text}"
        )


print("\nBearbeitung abgeschlossen.")

Feature-Class: Q:\SNAP\SNAP_GIS_Analyse\Adam_Analyse\FTDB_Bearbeitung\Dokom\Dokom.gdb\Unique_FeatureClass_Dokom
Quellfeld: TRASSEBESCHREIBUNG
Zielfeld: ROHR_TYP
Länge des Zielfeldes: 50
DN100/40-Regel: aktiviert

Extraktion wird gestartet ...
1000 Datensätze verarbeitet ...
2000 Datensätze verarbeitet ...
3000 Datensätze verarbeitet ...
4000 Datensätze verarbeitet ...
5000 Datensätze verarbeitet ...

QUALITÄTSKONTROLLE
1. Untersuchte Datensätze: 5039
2. Datensätze mit erkanntem Rohrtyp: 4280
   Davon erfolgreich gespeichert: 4280
3. Datensätze ohne erkannten Rohrtyp: 759
4. Datensätze mit mehreren Rohrtypen: 269

5. Alle unterschiedlichen erkannten Rohrtypen:
   - 32 mm
   - 40 mm
   - DN-100
   - DN-110
   - DN-125
   - DN-150
   - DN-160
   - DN-24
   - DN-25
   - DN-32
   - DN-40
   - DN-50
   - DN-63
   - DN-68
   - DN-75
   - FM-LWL-Leerrohr

6. Beispiele für nicht erkannte Beschreibungen:
   OID 1: Freie Vogelstr. 383
   OID 2: Faßstr. 1
   OID 5: Robert Schumann Str 8
   OID 6: 